In [1]:
from pypdf import PdfReader
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from bs4 import BeautifulSoup as bs
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import joblib

## Extract PDF file

In [2]:
reader = PdfReader("Python.pdf")

## Load Text format

In [3]:
text = ""
for page in reader.pages:
    text += page.extract_text()

In [4]:
print(text)

Python	Complete	Notes
From	Basics	to	Advanced	-	Explanation,	Syntax	and	Example	for	Every	Concept
Table	of	Contents
1.	Python	Basics	-	Variables	and	Data	Types
2.	Primitive	Data	Types
3.	Non-Primitive	Data	Types
4.	General	Built-in	Functions
5.	String	Built-in	Methods
6.	Membership	and	Identity	Operators
7.	Object-Oriented	Programming	(OOPs)
8.	Encapsulation	and	Bean	Class
9.	Inheritance	(is-a	Relation)
10.	Polymorphism
11.	Abstraction
12.	Singleton	Class
13.	Functions	(4	Types)	and	Inner	Functions
14.	Decorators
15.	File	Handling
16.	Exception	Handling
17.	Higher	Order	Functions
18.	Iterators
19.	Generators	(Function	and	Expression)
20.	Packing	and	Unpacking
21.	Comprehensions
22.	Database	Connection	(SQLite)
23.	JSON	and	Pickle	(Serialization	and	Deserialization)
24.	Regular	Expressions	(re	module)
25.	Additional	Important	Concepts
End	of	Notes
Table	of	Contents
1.	
Python	Basics	-	Variables	and	Data	Types
2.	
Primitive	Data	Types
3.	
Non-Primitive	Data	Types
4.	
General	Built-in	Fun

## NLP Text Normalization

In [5]:
def textNormalize(text):
    text = text.lower()
    text = bs(text,"html.parser").get_text()
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)
    text = text.translate(str.maketrans('','',string.punctuation))
    text = re.sub(r"\d+",'',text)
    text = re.sub(r'''[^\w\s.,;:!?''""\-]'''," ",text)
    text = re.sub(r"\s+",' ',text).strip()

    return text

In [6]:
text = textNormalize(text)

In [7]:
print(text)

python complete notes from basics to advanced explanation syntax and example for every concept table of contents python basics variables and data types primitive data types nonprimitive data types general builtin functions string builtin methods membership and identity operators objectoriented programming oops encapsulation and bean class inheritance isa relation polymorphism abstraction singleton class functions types and inner functions decorators file handling exception handling higher order functions iterators generators function and expression packing and unpacking comprehensions database connection sqlite json and pickle serialization and deserialization regular expressions re module additional important concepts end of notes table of contents python basics variables and data types primitive data types nonprimitive data types general builtin functions string builtin methods membership and identity operators objectoriented programming oops encapsulation and bean class inheritance 

## Recursive Chunk Split

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)

In [9]:
chapters = text.split("chapter")
chunks = []
for chapter in chapters:
    docs = text_splitter.create_documents([chapter])
    chunks.extend([doc.page_content for doc in docs])

print(len(chunks))

125


In [10]:
print(len(chunks))

125


In [11]:
for i in range(3):
    print(f"Chunk {i+1}")
    print(chunks[i])
    print("=" * 100)

Chunk 1
python complete notes from basics to advanced explanation syntax and example for every concept table of contents python basics variables and data types primitive data types nonprimitive data types general builtin functions string builtin methods membership and identity operators objectoriented programming oops encapsulation and bean class inheritance isa relation polymorphism abstraction singleton class functions types and inner functions decorators file handling exception handling higher order functions iterators generators function and expression packing and unpacking comprehensions database connection sqlite json and pickle serialization and deserialization regular expressions re module additional important concepts end of notes table of contents python basics variables and data types
Chunk 2
and deserialization regular expressions re module additional important concepts end of notes table of contents python basics variables and data types primitive data types nonprimitive d

## Embeddings

In [12]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
embeddings = embedding_model.encode(chunks,convert_to_numpy=True,show_progress_bar=True)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [14]:
print(embeddings.shape)

(125, 384)


## FAISS vector db

In [15]:
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(index.ntotal)

125


In [16]:
print(index.ntotal)

125


## store external file 

In [17]:
joblib.dump(chunks, "chunks.pkl")
faiss.write_index(index, "faiss_index.index")